In [1]:
import pandas as pd
from bertopic import BERTopic
from umap import UMAP
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import HDBSCAN
from stopwordsiso import stopwords
from multiprocessing import Pool
import ast
from tqdm.contrib.concurrent import process_map

import warnings
warnings.simplefilter("ignore")

/hpc/home/as1676/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
df = pd.read_csv('../../../data/comp_ideology_detection/interventions_us_with_embeddings.csv', index_col=0)
print(f"nrows: {df.shape[0]}")

df['embedding'] = process_map(ast.literal_eval, df['embedding'], max_workers=12, chunksize=5000)
    
docs = df['speech'].tolist()
embeddings = np.vstack(df['embedding'].values)

nrows: 822800


 57%|█████▋    | 465001/822800 [02:28<01:39, 3612.33it/s]

In [ ]:
umap_model = UMAP(n_jobs=12, random_state=123)
hdbscan_model = HDBSCAN(min_cluster_size=400, n_jobs=12)

topic_model = BERTopic(verbose=True,
                      umap_model=umap_model,
                      hdbscan_model=hdbscan_model,
                      vectorizer_model=CountVectorizer(stop_words=list(stopwords('en'))))

topics, probs = topic_model.fit_transform(docs, embeddings)

In [ ]:
topic_model.get_topic_info()